In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/minagaras/ai-gym-trainer-rag-docs/Evidence-BasedGuidelinesforResistanceTrainingVolumetoMaximizeMuscleHypertrophy.pdf
/kaggle/input/datasets/minagaras/ai-gym-trainer-rag-docs/jaeger-international-society-of-sports-nutrition-position-stand.pdf
/kaggle/input/datasets/minagaras/ai-gym-trainer-rag-docs/exercises.json


In [2]:
!pip install -q faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch bitsandbytes accelerate fastapi uvicorn pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 78.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 86.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 75.9 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingf

In [3]:
import os
import json
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

dataset_path = "/kaggle/input/datasets/minagaras/ai-gym-trainer-rag-docs"
documents = []

# 1. Load PDFs and JSON
for filename in os.listdir(dataset_path):
    file_path = os.path.join(dataset_path, filename)
    
    if filename.endswith(".pdf"):
        loader = PyPDFLoader(file_path)
        documents.extend(loader.load())
        print(f"Loaded PDF: {filename}")
        
    elif filename.endswith(".json"):
        with open(file_path, 'r', encoding='utf-8') as f:
            exercises = json.load(f)
            # Convert JSON entries into readable text documents for the LLM
            if isinstance(exercises, list):
                for ex in exercises:
                    content = json.dumps(ex, indent=2)
                    documents.append(Document(page_content=content, metadata={"source": filename}))
            else:
                documents.append(Document(page_content=json.dumps(exercises), metadata={"source": filename}))
        print(f"Loaded JSON: {filename}")

# 2. Split the text into manageable chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(chunks)}")

# 3. Generate Embeddings and Build FAISS Index
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)

print("Building FAISS database... (this may take a minute)")
vectordb = FAISS.from_documents(chunks, embedding)

# 4. Save the database locally to the Kaggle working directory
vectordb.save_local("faiss_gym_db")

/tmp/ipykernel_110/2398351782.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loaded PDF: Evidence-BasedGuidelinesforResistanceTrainingVolumetoMaximizeMuscleHypertrophy.pdf
Loaded PDF: jaeger-international-society-of-sports-nutrition-position-stand.pdf
Loaded JSON: exercises.json
Total chunks created: 41


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS database... (this may take a minute)


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Define the specific model ID
base_id = "mistralai/Mistral-Nemo-Instruct-2407"

# 2. Configure 4-bit quantization for Kaggle's GPU
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_id, use_fast=True)

# 4. Load the Model
model = AutoModelForCausalLM.from_pretrained(
    base_id, 
    quantization_config=bnb_cfg, 
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [5]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import json
import re

# 1. Define the Strict JSON Schema
response_schemas = [
    ResponseSchema(name="target_muscle_groups", description="List of target muscles for the routine"),
    ResponseSchema(name="weekly_split", description="The training split (e.g., Push/Pull/Legs)"),
    ResponseSchema(name="exercises", description="List of specific exercises with sets and reps"),
    ResponseSchema(name="form_cues", description="Specific form cues and corrective isolation movements (e.g., face pulls, serratus jabs)"),
    ResponseSchema(name="nutrition", description="Pre and post-workout nutrition options, strictly including homemade date and peanut oat snack bars")
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

# 2. Build the Prompt Template
gym_prompt_template = """
You are an expert AI Gym Trainer. Use the following academic context to generate a customized hypertrophy workout routine and nutrition plan.

Academic Context from FAISS:
{context}

User Profile:
- Age: {age}
- Weight: {weight}
- Goal: {goal}
- Equipment: {equipment}

Respond ONLY in JSON format as follows:
{format_instructions}
"""

prompt = PromptTemplate(
    template=gym_prompt_template,
    input_variables=["context", "age", "weight", "goal", "equipment", "format_instructions"]
)

print("Output Parser and Prompt Template initialized successfully!")
print("\nHere are the format instructions that will be injected into the LLM:")
print(format_instructions)

Output Parser and Prompt Template initialized successfully!

Here are the format instructions that will be injected into the LLM:
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"target_muscle_groups": string  // List of target muscles for the routine
	"weekly_split": string  // The training split (e.g., Push/Pull/Legs)
	"exercises": string  // List of specific exercises with sets and reps
	"form_cues": string  // Specific form cues and corrective isolation movements (e.g., face pulls, serratus jabs)
	"nutrition": string  // Pre and post-workout nutrition options, strictly including homemade date and peanut oat snack bars
}
```


In [6]:
import re

# 1. Define the LLM Generation Function
def generate_text(prompt_text, max_new_tokens=1000):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.7
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 2. Define the JSON Extractor
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return f"```json\n{matches[-1]}\n```"
    return text

# 3. The Main RAG Pipeline Function
def generate_gym_plan(age, weight, goal, equipment):
    print("Searching FAISS database for academic context...")
    query = f"hypertrophy volume for {goal} with {equipment}"
    docs = vectordb.similarity_search(query, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    print("Constructing prompt...")
    final_prompt = prompt.format(
        context=context,
        age=age,
        weight=weight,
        goal=goal,
        equipment=equipment,
        format_instructions=format_instructions
    )
    
    print("Generating AI response (this may take a minute)...")
    raw_response = generate_text(final_prompt)
    
    print("Extracting and parsing JSON...")
    clean_json = extract_json_block(raw_response)
    
    try:
        parsed_data = output_parser.parse(clean_json)
        return parsed_data
    except Exception as e:
        print("Parsing failed. Raw output was:")
        print(raw_response)
        return None

# 4. Test the Pipeline
test_plan = generate_gym_plan(
    age="22", 
    weight="75kg", 
    goal="Maximize chest and back hypertrophy", 
    equipment="Dumbbells, cables, leg press machine"
)

if test_plan:
    print("\nSUCCESS! Parsed Output:")
    print(json.dumps(test_plan, indent=2))

Searching FAISS database for academic context...
Constructing prompt...
Generating AI response (this may take a minute)...
Extracting and parsing JSON...

SUCCESS! Parsed Output:
{
  "target_muscle_groups": "Chest, Back",
  "weekly_split": "Push/Pull",
  "exercises": "Bench Press: 4 sets x 6-8 reps\nIncline Dumbbell Press: 3 sets x 8-12 reps\nWeighted Pull-ups: 4 sets x 6-8 reps\nBent-over Barbell Row: 3 sets x 8-12 reps",
  "form_cues": "Maintain full range of motion\nKeep shoulders retracted\nAvoid excessive arching of lower back",
  "nutrition": "Pre-workout: Oats with banana and peanut butter\nPost-workout: Homemade protein shake with homemade date and peanut oat snack bars"
}


In [7]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

app = FastAPI()

# Authentication Keys
NGROK_TOKEN = "3D2wFFE1eiQwQYUwL1KqxEPcFQV_284nV1nH3WgkJj7w9huRs" 
API_KEY = "secret123"

# 1. Define the API Route
@app.post("/generate")
async def gen(req: Request):
    # Verify the incoming request is authorized
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    
    data = await req.json()
    
    # Extract the user inputs sent from the local frontend
    age = data.get("age", "")
    weight = data.get("weight", "")
    goal = data.get("goal", "")
    equipment = data.get("equipment", "")
    
    # Trigger your RAG pipeline
    result = generate_gym_plan(age, weight, goal, equipment)
    
    return {"response": result}

# 2. Find an available port to avoid crashing Kaggle
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()

# 3. Open the Ngrok Tunnel
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("URL:", public_url + "/generate")

# 4. Start the Uvicorn server in a background thread
def run(): 
    uvicorn.run(app, host="0.0.0.0", port=port)

threading.Thread(target=run, daemon=True).start()
time.sleep(1)

URL: https://outing-kindness-iodine.ngrok-free.dev/generate                                         


INFO:     Started server process [110]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:38989 (Press CTRL+C to quit)


Searching FAISS database for academic context...
Constructing prompt...
Generating AI response (this may take a minute)...
Extracting and parsing JSON...
INFO:     41.39.36.230:0 - "POST /generate HTTP/1.1" 200 OK
